In [1]:
import os

In [2]:
os.chdir("../")
%pwd

'd:\\AI Projects\\Thesis project'

In [3]:
import pkgutil
import pypromice

modules = [m.name for m in pkgutil.iter_modules(pypromice.__path__)]
print(modules)

['core', 'io', 'pipeline', 'resources', 'tx']


In [4]:
import pandas as pd
from mlProject import logger


station = "KAN_L"
# We use the hourly URL that we know works perfectly
hourly_url = f"https://test-thredds.geus.dk/thredds/fileServer/aws/l2stations/csv/hour/{station}_hour.csv"

logger.info(f"Fetching hourly data for {station}...")

try:
    # 1. Load the hourly data
    df_hourly = pd.read_csv(hourly_url, 
                            comment='#', 
                            index_col=0, 
                            na_values=['', 'nan'], 
                            parse_dates=True)
    logger.info("Hourly data loaded successfully.")

    # 2. Convert to daily data! 
    # '1D' means 1 Day. .mean() calculates the daily average for every column.
    logger.info("Resampling hourly data to daily averages...")
    df_daily = df_hourly.resample('1D').mean()
    
    # 3. Check the results
    logger.info("Success! Here is a glance at the daily dataset:")
    with pd.option_context('display.max_columns', None, 'display.width', 1000):
        print(df_daily.head())
    
except Exception as e:
    logger.error(f"Failed to fetch or process data: {e}")

[2026-03-08 01:39:56,133: INFO: 1730775952: Fetching hourly data for KAN_L...]
[2026-03-08 01:40:04,636: INFO: 1730775952: Hourly data loaded successfully.]
[2026-03-08 01:40:04,636: INFO: 1730775952: Resampling hourly data to daily averages...]
[2026-03-08 01:40:04,675: INFO: 1730775952: Success! Here is a glance at the daily dataset:]
              rec         p_u       t_u       rh_u  rh_u_wrt_ice_or_water    wspd_u      wdir_u  wdir_std_u  wspd_x_u  wspd_y_u         dsr     dsr_cor        usr    usr_cor    albedo         dlr         ulr        cc  t_surf  z_boom_u  z_boom_cor_u  z_boom_q_u   z_stake  z_stake_cor   z_stake_q       z_pt   z_pt_cor  precip_u     t_i_1     t_i_2     t_i_3     t_i_4     t_i_5     t_i_6     t_i_7     t_i_8    tilt_x    tilt_y  rot    gps_lat    gps_lon     gps_alt       gps_time  gps_geoid  gps_geounit  gps_hdop  gps_q     batt_v  batt_v_ss    fan_dc_u     t_log     t_rad        p_i       t_i       rh_i  rh_i_wrt_ice_or_water    wspd_i      wdir_i  wspd_

In [5]:
from dataclasses import dataclass
from pathlib import Path
from typing import List

@dataclass(frozen=True)
class DataIngestionConfig:
    root_dir: Path
    base_url: str
    stations: List[str]
    local_data_dir: Path

In [6]:
from mlProject.constants import *
from mlProject.utils.common import read_yaml, create_directories

class ConfigurationManager:
    def __init__(
        self,
        config_filepath=CONFIG_FILE_PATH,
        params_filepath=PARAMS_FILE_PATH,
        schema_filepath=SCHEMA_FILE_PATH,
    ):
        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)
        self.schema = read_yaml(schema_filepath)
        create_directories([Path(self.config.artifacts_root)])

    def get_data_ingestion_config(self) -> DataIngestionConfig:
        config = self.config.data_ingestion

        # Create both the root dir and the specific folder for the daily CSVs
        create_directories([config.root_dir, config.local_data_dir])

        data_ingestion_config = DataIngestionConfig(
            root_dir=Path(config.root_dir),
            base_url=config.base_url,
            stations=list(config.stations),
            local_data_dir=Path(config.local_data_dir)
        )

        return data_ingestion_config

In [7]:
import pandas as pd
from mlProject import logger

class DataIngestion:
    def __init__(self, config: DataIngestionConfig):
        self.config = config

    def download_and_resample_to_daily(self):
        """
        Fetches hourly data for all stations defined in config,
        resamples to daily averages, and saves as local CSVs.
        """
        for station in self.config.stations:
            # Construct the hourly URL dynamically
            target_url = f"{self.config.base_url}hour/{station}_hour.csv"
            
            # Construct where the daily file will be saved
            local_file_path = Path(self.config.local_data_dir) / f"{station}_daily.csv"
            
            logger.info(f"Fetching hourly data for {station}...")
            
            try:
                # 1. Load the hourly data
                df_hourly = pd.read_csv(
                    target_url, 
                    comment='#', 
                    index_col=0, 
                    na_values=['', 'nan'], 
                    parse_dates=True
                )
                
                # 2. Resample to daily averages
                logger.info(f"Resampling {station} to daily averages...")
                df_daily = df_hourly.resample('1D').mean()
                
                # 3. Save the daily data locally
                df_daily.to_csv(local_file_path)
                logger.info(f"Successfully saved {station} daily data to {local_file_path}")
                
            except Exception as e:
                logger.error(f"Failed processing data for {station}: {e}")
                raise e

In [10]:
try:
    config = ConfigurationManager()
    data_ingestion_config = config.get_data_ingestion_config()
    
    data_ingestion = DataIngestion(config=data_ingestion_config)
    data_ingestion.download_and_resample_to_daily()
    
except Exception as e:
    logger.error(e)
    raise e

[2026-03-08 01:42:13,783: INFO: common: yaml file: config\config.yaml loaded successfully]
[2026-03-08 01:42:13,785: INFO: common: yaml file: params.yaml loaded successfully]
[2026-03-08 01:42:13,786: INFO: common: yaml file: schema.yaml loaded successfully]
[2026-03-08 01:42:13,786: INFO: common: created directory at: artifacts]
[2026-03-08 01:42:13,786: INFO: common: created directory at: artifacts/data_ingestion]
[2026-03-08 01:42:13,829: INFO: common: created directory at: artifacts/data_ingestion/daily_data]
[2026-03-08 01:42:13,830: INFO: 838429704: Fetching hourly data for KAN_L...]
[2026-03-08 01:42:28,844: INFO: 838429704: Resampling KAN_L to daily averages...]
[2026-03-08 01:42:29,289: INFO: 838429704: Successfully saved KAN_L daily data to artifacts\data_ingestion\daily_data\KAN_L_daily.csv]
[2026-03-08 01:42:29,290: INFO: 838429704: Fetching hourly data for EGP...]
[2026-03-08 01:42:35,479: INFO: 838429704: Resampling EGP to daily averages...]
[2026-03-08 01:42:35,744: INFO